In [1]:
import pandas as pd
import duckdb

df_device_status_log = pd.DataFrame({
    "record_id": [
        101, 102, 103, 104, 105, 106,
        201, 202, 203, 204, 205,
        301, 302, 303, 304, 305
    ],
    "device_id": [
        "R05", "R05", "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34", "R34"
    ],
    "record_time": [
        "2026-08-02 08:00:00",
        "2026-08-02 08:10:00",
        "2026-08-02 08:20:00",
        "2026-08-02 08:30:00",
        "2026-08-02 08:40:00",
        "2026-08-02 08:50:00",

        "2026-08-02 09:00:00",
        "2026-08-02 09:10:00",
        "2026-08-02 09:20:00",
        "2026-08-02 09:30:00",
        "2026-08-02 09:40:00",

        "2026-08-02 10:00:00",
        "2026-08-02 10:10:00",
        "2026-08-02 10:20:00",
        "2026-08-02 10:30:00",
        "2026-08-02 10:40:00"
    ],
    "status": [
        "NORMAL", "NORMAL", "ERROR", "ERROR", "NORMAL", "NORMAL",
        "NORMAL", "WARNING", "WARNING", "ERROR", "NORMAL",
        "ERROR", "ERROR", "NORMAL", "ERROR", "ERROR"
    ]
})

df_device_status_log["record_time"] = pd.to_datetime(
    df_device_status_log["record_time"]
)

df_device_status_log

,record_id,device_id,record_time,status
0,101,R05,2026-08-02 08:00:00,NORMAL
1,102,R05,2026-08-02 08:10:00,NORMAL
2,103,R05,2026-08-02 08:20:00,ERROR
3,104,R05,2026-08-02 08:30:00,ERROR
4,105,R05,2026-08-02 08:40:00,NORMAL
5,106,R05,2026-08-02 08:50:00,NORMAL
6,201,R16,2026-08-02 09:00:00,NORMAL
7,202,R16,2026-08-02 09:10:00,WARNING
8,203,R16,2026-08-02 09:20:00,WARNING
9,204,R16,2026-08-02 09:30:00,ERROR


# SQL Daily Review：识别设备状态变化点

## 题目背景

设备会持续产生状态记录。

状态包括：

- `NORMAL`
- `WARNING`
- `ERROR`

现在需要按照时间顺序比较当前状态和上一条状态，识别设备状态发生变化的记录。

---

## 题目要求

对于每台设备的每条记录：

1. 获取上一条记录的状态；
2. 判断当前状态与上一条状态是否不同；
3. 只输出**状态发生变化的记录**。

---

## 状态变化规则

如果：

```text
当前 status != previous_status
```

则认为发生了状态变化。

例如：

```text
08:00  NORMAL
08:10  NORMAL
08:20  ERROR
08:30  ERROR
08:40  NORMAL
```

状态变化发生在：

```text
08:20  NORMAL → ERROR
08:40  ERROR  → NORMAL
```

因此只输出 `08:20` 和 `08:40`。

---

## 第一条记录的处理

每台设备的第一条记录没有上一条状态：

```text
previous_status = NULL
```

第一条记录**不视为状态变化**，因此不要输出。

---

## 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `record_id` | 当前记录编号 |
| `record_time` | 当前记录时间 |
| `previous_status` | 上一条记录状态 |
| `status` | 当前记录状态 |

---

## 记录顺序

每台设备内部按照：

1. `record_time` 升序；
2. `record_id` 升序。

确定前后记录关系。

---

## 最终排序

按照：

1. `device_id` 升序；
2. `record_time` 升序；
3. `record_id` 升序。

---

## 解题要求

- 使用 `LAG()` 获取上一条状态；
- 使用 `PARTITION BY device_id`；
- 窗口排序同时使用 `record_time` 和 `record_id`；
- 使用 CTE；
- 外层查询只保留状态发生变化的记录；
- 第一条记录不能被识别为状态变化；
- 不使用自连接。

In [6]:
query = """
WITH previous_status_table AS (
    SELECT
        device_id,
        record_id,
        record_time,
        status,
        LAG(status) OVER (
            PARTITION BY device_id
            ORDER BY record_time, record_id
        ) AS previous_status
    FROM df_device_status_log
)

SELECT
    device_id,
    record_id,
    record_time,
    previous_status,
    status
FROM previous_status_table
WHERE previous_status IS NOT NULL
  AND status != previous_status
ORDER BY
    device_id,
    record_time,
    record_id;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,record_id,record_time,previous_status,status
0,R05,103,2026-08-02 08:20:00,NORMAL,ERROR
1,R05,105,2026-08-02 08:40:00,ERROR,NORMAL
2,R16,202,2026-08-02 09:10:00,NORMAL,WARNING
3,R16,204,2026-08-02 09:30:00,WARNING,ERROR
4,R16,205,2026-08-02 09:40:00,ERROR,NORMAL
5,R34,303,2026-08-02 10:20:00,ERROR,NORMAL
6,R34,304,2026-08-02 10:30:00,NORMAL,ERROR
